In [ ]:
def analyser_requete_mistral(requete, perimetre):
    prompt = f"""
Tu es un moteur NLP avancé intégré à un système ERP intelligent. Tu transformes les requêtes utilisateurs en structure JSON exploitable par un moteur SQL.

Voici le périmètre fonctionnel (tables et colonnes disponibles) :
{json.dumps(perimetre, indent=2, ensure_ascii=False)}

== Requête utilisateur ==
"{requete}"

== Objectif ==
Génère un JSON contenant :
- "requete_originale" : la phrase d’origine.
- "intention" : l’action demandée (ex: "lister_commandes", "voir_total", "lister_clients", "lister_produits", "lister_fournisseurs")
- "table" : nom de la table cible (parmi celles du périmètre)
- "filtres" : dictionnaire de paires clé/valeur pertinentes (client, pays, date, plage_date, catégorie, produit...)

== Instructions ==
- Si un mois + année est détecté (ex: "avril 2024"), renvoie-le comme : "date": "avril 2024"
- Si une **plage de dates** est mentionnée (ex: "janvier à mars 2023"), structure comme :
  "plage_date": {{
    "date_debut": "janvier 2023",
    "date_fin": "mars 2023"
  }}
- Reste strictement dans le périmètre fourni
- Formate uniquement le JSON demandé, sans aucun commentaire ni explication

== Format attendu ==
{{
  "requete_originale": "...",
  "intention": "...",
  "table": "...",
  "filtres": {{
    "clé": "valeur"
  }}
}}
"""

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": "llama3.2", "prompt": prompt, "stream": False}
        )
        raw = response.json()["response"].strip()

        # 🧼 Nettoyer le contenu s’il est encapsulé dans des ```
        if raw.startswith("```"):
            raw = raw.split("```")[1].strip()

        return json.loads(raw)

    except Exception as e:
        print("❌ Erreur JSON :", e)
        print("🧾 Réponse brute du modèle :\n", raw)
        return None


# === Données de test ===
perimetre = {
    "commandes": ["id", "client", "produit", "montant", "date", "fournisseur", "categorie", "pays"],
    "clients": ["id", "nom", "secteur", "pays"],
    "produits": ["id", "nom", "categorie", "disponibilite"],
    "fournisseurs": ["id", "nom", "pays"]
}

requêtes = [
    "Donne-moi les commandes de Involys passées en avril 2024 pour le produit logiciel RH",
    "Liste des clients du secteur public ayant commandé en 2023",
    "Quel est le total des ventes en mai 2022 ?",
    "Quels sont les fournisseurs marocains ?",
    "Montre-moi les produits commandés par Maroc Telecom entre janvier et mars 2024",
    "Combien de commandes ont été faites en 2024 ?",
    "Voir tous les clients ayant acheté des articles dans la catégorie éducation",
    "Afficher les montants des commandes de mars",
    "Commandes faites par les clients internationaux",
    "Quels sont les produits disponibles ?"
]

for i, req in enumerate(requêtes, 1):
    print(f"\n===== Requête {i} =====")
    resultat = analyser_requete_mistral(req, perimetre)
    print(json.dumps(resultat, indent=4, ensure_ascii=False))
